# E-Commerce Revenue Leakage & Profitability Analysis

## Data Understanding & Data Quality Analysis

### Project Objective
This project analyzes a synthetic e-commerce dataset to identify potential sources of revenue and profit leakage, including discounts, returns, cancellations, shipping costs, delivery delays, and low-margin products.

### Dataset
- Synthetic e-commerce transaction dataset
- Approximately 100,000 orders
- No real company or customer data is used

### Current Analysis
This notebook focuses on:
- Dataset structure
- Data types
- Missing values
- Duplicate records
- Unique identifiers
- Date coverage
- Numerical distributions
- Categorical distributions
- Order-status validation
- Basic business sanity checks

In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}") 

In [2]:
file_path = "../data/ecommerce_revenue_leakage_100k.csv"
df = pd.read_csv(file_path)
print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 100000
Number of columns: 31


In [4]:
print("Dataset Columns:\n")
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

Dataset Columns:

1. order_id
2. order_date
3. customer_id
4. customer_segment
5. product_id
6. product_name
7. category
8. sub_category
9. region
10. state
11. quantity
12. unit_price
13. gross_revenue
14. discount_pct
15. discount_amount
16. sales
17. unit_cost
18. cogs
19. shipping_cost
20. delivery_days
21. payment_method
22. order_status
23. returned
24. return_reason
25. cancellation_loss
26. return_loss
27. net_revenue
28. net_profit
29. profit_margin_pct
30. revenue_leakage
31. leakage_reason


In [5]:
df.head()

,order_id,order_date,customer_id,customer_segment,product_id,product_name,category,sub_category,region,state,quantity,unit_price,gross_revenue,discount_pct,discount_amount,sales,unit_cost,cogs,shipping_cost,delivery_days,payment_method,order_status,returned,return_reason,cancellation_loss,return_loss,net_revenue,net_profit,profit_margin_pct,revenue_leakage,leakage_reason
0,ORD001488,2024-01-01,CUST10525,Corporate,PRD0546,Smartphones Item 028,Electronics,Smartphones,North,Delhi,2,"37,863.18","75,726.36",20,"15,145.27","60,581.09","31,015.01","62,030.02",217.45,6,COD,Completed,0,NaN,0.00,0.00,"60,581.09","-1,666.38",-2.75,"15,362.72",Low-Margin Product
1,ORD002259,2024-01-01,CUST01805,Consumer,PRD2141,Women Clothing Item 183,Fashion,Women Clothing,North,Delhi,1,"1,436.08","1,436.08",25,359.02,"1,077.06",633.94,633.94,212.72,6,UPI,Completed,0,NaN,0.00,0.00,"1,077.06",230.40,21.39,571.74,Healthy Order
2,ORD002316,2024-01-01,CUST05345,Consumer,PRD1595,Furniture Item 037,Home & Furniture,Furniture,West,Goa,1,"6,778.94","6,778.94",25,"1,694.74","5,084.20","4,417.68","4,417.68",67.87,5,Debit Card,Completed,0,NaN,0.00,0.00,"5,084.20",598.65,11.77,"1,762.61",Healthy Order
3,ORD002442,2024-01-01,CUST08697,Consumer,PRD1788,Men Clothing Item 003,Fashion,Men Clothing,West,Maharashtra,3,"3,637.86","10,913.59",10,"1,091.36","9,822.23","1,666.41","4,999.23",243.03,3,UPI,Completed,0,NaN,0.00,0.00,"9,822.23","4,579.97",46.63,"1,334.39",Healthy Order
4,ORD003674,2024-01-01,CUST17746,Consumer,PRD1715,Kitchen Item 220,Home & Furniture,Kitchen,West,Goa,2,"1,959.11","3,918.22",0,0.00,"3,918.22","1,096.91","2,193.82",179.36,3,UPI,Completed,0,NaN,0.00,0.00,"3,918.22","1,545.04",39.43,179.36,Healthy Order


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 31 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   order_id           100000 non-null  str    
 1   order_date         100000 non-null  str    
 2   customer_id        100000 non-null  str    
 3   customer_segment   100000 non-null  str    
 4   product_id         100000 non-null  str    
 5   product_name       100000 non-null  str    
 6   category           100000 non-null  str    
 7   sub_category       100000 non-null  str    
 8   region             100000 non-null  str    
 9   state              100000 non-null  str    
 10  quantity           100000 non-null  int64  
 11  unit_price         100000 non-null  float64
 12  gross_revenue      100000 non-null  float64
 13  discount_pct       100000 non-null  int64  
 14  discount_amount    100000 non-null  float64
 15  sales              100000 non-null  float64
 16  unit_cost     

In [7]:
df.dtypes.to_frame(name="data_type")

,data_type
order_id,str
order_date,str
customer_id,str
customer_segment,str
product_id,str
product_name,str
category,str
sub_category,str
region,str
state,str


In [8]:
missing_values = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df)) * 100})
missing_values

,missing_count,missing_percentage
order_id,0,0.00
order_date,0,0.00
customer_id,0,0.00
customer_segment,0,0.00
product_id,0,0.00
product_name,0,0.00
category,0,0.00
sub_category,0,0.00
region,0,0.00
state,0,0.00


In [9]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 91867


In [10]:
duplicate_rows = df.duplicated().sum()
print("Duplicate rows:", duplicate_rows)

Duplicate rows: 0


In [11]:
duplicate_order_ids = df["order_id"].duplicated().sum()
print("Duplicate Order IDs:", duplicate_order_ids)

Duplicate Order IDs: 0


## 3.2 Unique Identifiers, Date Range & Descriptive Statistics

This section validates key business identifiers, checks the dataset's time coverage, and summarizes the numerical variables.

In [12]:
print("Total Orders:", df["order_id"].nunique())
print("Unique Customers:", df["customer_id"].nunique())
print("Unique Products:", df["product_id"].nunique())

Total Orders: 100000
Unique Customers: 19872
Unique Products: 2500


In [13]:
df["order_date"] = pd.to_datetime(df["order_date"])
print("Order Date Data Type:", df["order_date"].dtype)

Order Date Data Type: datetime64[us]


In [14]:
print("Start Date:", df["order_date"].min())
print("End Date:", df["order_date"].max())

Start Date: 2024-01-01 00:00:00
End Date: 2025-12-31 00:00:00


In [15]:
date_range_days = (df["order_date"].max() - df["order_date"].min()).days
print("Dataset Period (Days):", date_range_days)

Dataset Period (Days): 730


In [16]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
order_date,100000,2024-12-30 21:31:14.016000,2024-01-01 00:00:00,2024-07-01 00:00:00,2025-01-01 00:00:00,2025-07-01 00:00:00,2025-12-31 00:00:00,NaN
quantity,"100,000.00",1.71,1.00,1.00,1.00,2.00,5.00,1.03
unit_price,"100,000.00","11,553.44",250.38,"1,991.76","3,761.24","10,575.01","84,996.94","17,755.46"
gross_revenue,"100,000.00","19,787.87",250.38,"2,653.10","5,271.04","17,621.82","422,813.44","37,533.02"
discount_pct,"100,000.00",15.43,0.00,10.00,15.00,20.00,40.00,10.55
discount_amount,"100,000.00","3,036.53",0.00,219.24,694.75,"2,212.81","164,802.33","7,178.78"
sales,"100,000.00","16,751.34",165.56,"2,215.07","4,459.05","14,917.08","422,813.44","32,159.37"
unit_cost,"100,000.00","8,490.82",105.14,959.57,"1,895.86","7,168.78","73,733.03","14,660.95"
cogs,"100,000.00","14,549.24",105.14,"1,298.05","2,679.11","10,998.22","362,982.15","30,656.31"
shipping_cost,"100,000.00",210.27,60.33,152.94,205.99,256.82,608.61,75.17


In [17]:
business_metrics = [
    "quantity",
    "unit_price",
    "gross_revenue",
    "discount_pct",
    "discount_amount",
    "sales",
    "cogs",
    "shipping_cost",
    "delivery_days",
    "net_revenue",
    "net_profit",
    "profit_margin_pct",
    "revenue_leakage"
]
df[business_metrics].describe().T

,count,mean,std,min,25%,50%,75%,max
quantity,"100,000.00",1.71,1.03,1.00,1.00,1.00,2.00,5.00
unit_price,"100,000.00","11,553.44","17,755.46",250.38,"1,991.76","3,761.24","10,575.01","84,996.94"
gross_revenue,"100,000.00","19,787.87","37,533.02",250.38,"2,653.10","5,271.04","17,621.82","422,813.44"
discount_pct,"100,000.00",15.43,10.55,0.00,10.00,15.00,20.00,40.00
discount_amount,"100,000.00","3,036.53","7,178.78",0.00,219.24,694.75,"2,212.81","164,802.33"
sales,"100,000.00","16,751.34","32,159.37",165.56,"2,215.07","4,459.05","14,917.08","422,813.44"
cogs,"100,000.00","14,549.24","30,656.31",105.14,"1,298.05","2,679.11","10,998.22","362,982.15"
shipping_cost,"100,000.00",210.27,75.17,60.33,152.94,205.99,256.82,608.61
delivery_days,"100,000.00",4.11,1.64,1.00,3.00,4.00,5.00,10.00
net_revenue,"100,000.00","14,935.69","30,836.15",0.00,"1,533.53","3,788.79","12,172.17","422,813.44"


## 3.3 Categorical Distributions & Order Status Validation

This section examines the distribution of key categorical variables and validates the consistency of order-status-related fields.

In [18]:
categorical_columns = [
    "customer_segment",
    "category",
    "sub_category",
    "region",
    "payment_method",
    "order_status",
    "return_reason",
    "leakage_reason"
]
for column in categorical_columns:
    print(f"\n--- {column.upper()} ---")
    print(df[column].value_counts(dropna=False))


--- CUSTOMER_SEGMENT ---
customer_segment
Consumer       65934
Corporate      19913
Home Office    14153
Name: count, dtype: int64

--- CATEGORY ---
category
Fashion             29901
Electronics         27925
Home & Furniture    27260
Beauty              14914
Name: count, dtype: int64

--- SUB_CATEGORY ---
sub_category
Men Clothing      9981
Footwear          9968
Women Clothing    9952
Accessories       9451
Smartphones       9272
Laptops           9202
Furniture         9138
Kitchen           9132
Home Decor        8990
Hair Care         4994
Skin Care         4989
Fragrances        4931
Name: count, dtype: int64

--- REGION ---
region
West     30049
South    28026
North    23940
East     17985
Name: count, dtype: int64

--- PAYMENT_METHOD ---
payment_method
UPI            38159
Credit Card    21907
Debit Card     16187
COD            13788
Net Banking     9959
Name: count, dtype: int64

--- ORDER_STATUS ---
order_status
Completed    88644
Returned      8133
Cancelled     3223
Nam

In [19]:
order_status_counts = df["order_status"].value_counts()
print("Order Status Distribution:")
print(order_status_counts)
print("\nOrder Status Percentage:")
print((df["order_status"].value_counts(normalize=True) * 100).round(2))

Order Status Distribution:
order_status
Completed    88644
Returned      8133
Cancelled     3223
Name: count, dtype: int64

Order Status Percentage:
order_status
Completed   88.64
Returned     8.13
Cancelled    3.22
Name: proportion, dtype: float64


In [20]:
pd.crosstab(
    df["order_status"],
    df["returned"],
    margins=True
)

returned,0,1,All
order_status,,,
Cancelled,3223,0,3223
Completed,88644,0,88644
Returned,0,8133,8133
All,91867,8133,100000


In [21]:
print(
    "Returned orders without return reason:",
    df.loc[
        df["order_status"] == "Returned",
        "return_reason"
    ].isna().sum()
)
print(
    "Non-returned orders with return reason:",
    df.loc[
        df["order_status"] != "Returned",
        "return_reason"
    ].notna().sum()
)

Returned orders without return reason: 0
Non-returned orders with return reason: 0


## 3.4 Basic Business Sanity Checks

This section validates whether key financial and operational variables contain logically valid values before proceeding to exploratory analysis.

In [22]:
print("Quantity <= 0:", (df["quantity"] <= 0).sum())
print("Unit Price <= 0:", (df["unit_price"] <= 0).sum())
print("Gross Revenue <= 0:", (df["gross_revenue"] <= 0).sum())

Quantity <= 0: 0
Unit Price <= 0: 0
Gross Revenue <= 0: 0


In [23]:
print("Discount < 0%:", (df["discount_pct"] < 0).sum())
print("Discount > 100%:", (df["discount_pct"] > 100).sum())

print("\nMinimum Discount:", df["discount_pct"].min())
print("Maximum Discount:", df["discount_pct"].max())

Discount < 0%: 0
Discount > 100%: 0

Minimum Discount: 0
Maximum Discount: 40


In [24]:
print("COGS < 0:", (df["cogs"] < 0).sum())
print("Shipping Cost < 0:", (df["shipping_cost"] < 0).sum())
print("Discount Amount < 0:", (df["discount_amount"] < 0).sum())

COGS < 0: 0
Shipping Cost < 0: 0
Discount Amount < 0: 0


In [25]:
print("Delivery Days < 0:", (df["delivery_days"] < 0).sum())
print("Minimum Delivery Days:", df["delivery_days"].min())
print("Maximum Delivery Days:", df["delivery_days"].max())

Delivery Days < 0: 0
Minimum Delivery Days: 1
Maximum Delivery Days: 10


In [26]:
gross_revenue_check = np.isclose(
    df["gross_revenue"],
    df["quantity"] * df["unit_price"],
    atol=0.02
)
print(
    "Gross Revenue Formula Mismatches:",
    (~gross_revenue_check).sum()
)

Gross Revenue Formula Mismatches: 0


In [27]:
discount_check = np.isclose(
    df["discount_amount"],
    df["gross_revenue"] * df["discount_pct"] / 100,
    atol=0.02
)

print(
    "Discount Amount Formula Mismatches:",
    (~discount_check).sum()
)

Discount Amount Formula Mismatches: 0


In [28]:
loss_making_orders = (df["net_profit"] < 0).sum()
loss_making_pct = (loss_making_orders / len(df)) * 100

print("Loss-Making Orders:", loss_making_orders)
print("Loss-Making Orders %:", round(loss_making_pct, 2))

Loss-Making Orders: 19761
Loss-Making Orders %: 19.76


In [29]:
zero_net_revenue = (df["net_revenue"] == 0).sum()
print("Orders with Zero Net Revenue:", zero_net_revenue)

Orders with Zero Net Revenue: 11356


## 3.5 Order Status & Financial Consistency Validation

This section validates whether zero-revenue orders and financial loss fields are logically consistent with order status.

In [30]:
zero_revenue_by_status = df.loc[df["net_revenue"] == 0,"order_status"].value_counts()
print("Zero Net Revenue Orders by Status:")
print(zero_revenue_by_status)

Zero Net Revenue Orders by Status:
order_status
Returned     8133
Cancelled    3223
Name: count, dtype: int64


In [31]:
print(
    "Returned orders with zero return loss:",
    (
        (df["order_status"] == "Returned") &
        (df["return_loss"] == 0)
    ).sum()
)
print(
    "Non-returned orders with return loss:",
    (
        (df["order_status"] != "Returned") &
        (df["return_loss"] > 0)
    ).sum()
)

Returned orders with zero return loss: 0
Non-returned orders with return loss: 0


In [32]:
print(
    "Cancelled orders with zero cancellation loss:",
    (
        (df["order_status"] == "Cancelled") &
        (df["cancellation_loss"] == 0)
    ).sum()
)
print(
    "Non-cancelled orders with cancellation loss:",
    (
        (df["order_status"] != "Cancelled") &
        (df["cancellation_loss"] > 0)
    ).sum()
)

Cancelled orders with zero cancellation loss: 0
Non-cancelled orders with cancellation loss: 0


In [ ]:
negative_profit_by_status = df.loc[df["net_profit"] < 0,"order_status"].value_counts()
print("Loss-Making Orders by Status:")
print(negative_profit_by_status)

Loss-Making Orders by Status:
order_status
Completed    8405
Returned     8133
Cancelled    3223
Name: count, dtype: int64


: 

## Step 3 Summary — Data Understanding & Quality Assessment

- Dataset contains 100,000 orders and 31 columns.
- 100,000 unique order IDs were identified with no duplicate order IDs.
- Dataset contains 19,872 unique customers and 2,500 unique products.
- The dataset covers transactions from January 1, 2024 to December 31, 2025.
- No duplicate rows were found.
- `return_reason` contains 91,867 null values; however, these are structurally expected because only returned orders require a return reason.
- All 8,133 returned orders contain a valid return reason.
- Order status distribution:
  - Completed: 88,644
  - Returned: 8,133
  - Cancelled: 3,223
- Order status and returned flags are fully consistent.
- Quantity, pricing, discount, cost, shipping, and delivery-day values passed basic validity checks.
- Discount values range from 0% to 40%.
- Delivery time ranges from 1 to 10 days.
- Gross revenue and discount amount calculations showed zero formula mismatches.
- 19,761 orders (19.76%) are loss-making.
- Loss-making orders include:
  - Completed: 8,405
  - Returned: 8,133
  - Cancelled: 3,223
- 11,356 orders have zero net revenue, consisting exactly of:
  - Returned: 8,133
  - Cancelled: 3,223
- Return-loss and cancellation-loss fields are consistent with their respective order statuses.

### Initial Business Observation

A notable finding is that 8,405 completed orders are loss-making. This indicates that profitability leakage is not limited to returns and cancellations. Further exploratory analysis should investigate the impact of discounts, shipping costs, product margins, categories, regions, and other operational factors on completed-order profitability.

### Data Quality Conclusion

The synthetic dataset is structurally consistent and suitable for further exploratory and business analysis. No records have been removed or modified during this data-understanding stage.